In [26]:
!unzip "/content/archive.zip" -d "/content/mrbeast_sentiment_data"

Archive:  /content/archive.zip
replace /content/mrbeast_sentiment_data/sentiment_analysis_dataset.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [1]:
import os
import pandas as pd
import re
import string
from nltk.corpus import stopwords
import  nltk
from collections import Counter

import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, TensorDataset
from tqdm import trange
import matplotlib.pyplot as plt

import numpy as np

In [2]:
extracted_dir = "/content/mrbeast_sentiment_data"
if os.path.exists(extracted_dir):
    print(os.listdir(extracted_dir))
else:
    print(f"Directory {extracted_dir} not found. Please check the unzip output.")

['sentiment_analysis_dataset.csv']


In [3]:
path = "/content/mrbeast_sentiment_data/sentiment_analysis_dataset.csv"

reviews = pd.read_csv(path,
                      engine='python',
                      quotechar='"',
                      sep=',',
                      on_bad_lines='skip')
reviews.dropna(axis=0, inplace=True)

In [4]:
nltk.download('stopwords')
stop_words = stopwords.words('english')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
def process_text(text):
    text = text.lower()
    text = re.sub(r'https?://[^\s\n\r]+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = [word for word in text.split() if word not in stop_words and
             word not in string.punctuation]
    return " ".join(words)

In [6]:
print(process_text(reviews["Comment"][0]))
print(reviews["Comment"][0])

achieving million views days dangerous
Achieving million views in days is dangerous


In [7]:
reviews["Comment"] = reviews["Comment"].map(lambda x: process_text(x))
reviews["Comment"].head()

,Comment
0,achieving million views days dangerous
1,many people want participate challenges?
2,mrbeast slowly turning mrjigsaw
3,genuinely can't believe dystopian
4,worlds smartest people compete series intellec...


In [8]:
reviews.loc[reviews["Sentiment"] == "Positive", "Sentiment"] = 0
reviews.loc[reviews["Sentiment"] == "Neutral", "Sentiment"] = 1
reviews.loc[reviews["Sentiment"] == "Negative", "Sentiment"] = 2

In [9]:
counter = Counter()

for comment in reviews["Comment"]:
    counter.update(comment.split())

# elimina palavras muito raras
min_freq = 2
counter = {w: c for w, c in counter.items() if c >= min_freq}

# monta vocabulário ordenado do mais frequente ao menos frequente
words = {
    "<pad>": 0,
    "<unk>": 1,
}

for w, _ in sorted(counter.items(), key=lambda x: -x[1]):
    words[w] = len(words)

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [11]:
def tokenized(text):
    tokens = []
    for word in text.split():
        idx = words.get(word, words["<unk>"])
        tokens.append(idx)
    return torch.tensor(tokens, dtype=torch.long).to(device)

In [12]:
tokenized(reviews["Comment"][0])

tensor([874,  17, 166,   9, 875])

In [13]:
X = reviews["Comment"].apply(tokenized).to_list()
y = torch.tensor(reviews["Sentiment"], dtype=torch.long).to(device)

In [14]:
def collate_batch(batch):
    texts, labels = zip(*batch)
    texts = pad_sequence(texts, batch_first=True, padding_value=words["<pad>"])
    labels = torch.tensor(labels, dtype=torch.long)
    return texts.to(device), labels.to(device)

In [15]:
dataset = list(zip(X, y))
loader = DataLoader(dataset, batch_size=32,
                    collate_fn=collate_batch, shuffle=True)

In [16]:
def load_glove(path):
    embed_dict = {}
    with open(path, "r", encoding="utf8") as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], dtype="float32")
            embed_dict[word] = vector
    return embed_dict

In [17]:
glove = load_glove("glove.6B.100d.txt")
embedding_dim = 100
vocab_size = len(words)

embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, idx in words.items():
    vector = glove.get(word)
    if vector is not None:
        embedding_matrix[idx] = vector
    else:
        if word == "<pad>":
            embedding_matrix[idx] = np.zeros(embedding_dim)
        else:
            embedding_matrix[idx] = np.random.normal(scale=0.6, size=(embedding_dim,))

embedding_matrix = torch.tensor(embedding_matrix, dtype=torch.float32)

In [18]:
class SentimentAnalysis(nn.Module):
    def __init__(self, vocab_size, n_classes, pad_idx=0, embed_dim=100):
        super().__init__()

        self.embedding = nn.Embedding.from_pretrained(
            embeddings=embedding_matrix,
            freeze=True
        )

        self.r1 = nn.LSTM(embed_dim, 128, batch_first=True)
        self.r2 = nn.LSTM(128, 64, batch_first=True)

        # camada de atenção: transforma hidden -> score escalar
        self.attention = nn.Linear(64, 1)

        self.fc = nn.Linear(64, n_classes)

    def forward(self, x):
      mask = (x != words["<pad>"]).unsqueeze(-1).float()   # (batch, seq, 1)

      x = self.embedding(x)

      x, _ = self.r1(x)
      x = F.dropout(x, 0.5)

      x, _ = self.r2(x)
      x = F.dropout(x, 0.5)

      scores = self.attention(x)   # (batch, seq, 1)

      # mascara PAD penalizando com -inf
      scores = scores = scores.masked_fill(mask == 0, -1e9)

      attn_weights = F.softmax(scores, dim=1)

      context = torch.sum(attn_weights * x, dim=1)

      return self.fc(context)

In [19]:
n_classes = 3
LR = 2e-4
model = SentimentAnalysis(vocab_size, n_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

In [20]:
epochs = 30
losses = []
accs = []

for epoch in trange(epochs, desc="Training"):
    total_loss = 0
    acc = 0

    for batch, (x, y) in enumerate(loader):
        optimizer.zero_grad()

        pred = model(x)
        loss = criterion(pred, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        _, predicted = pred.max(1)
        acc += (predicted == y).float().sum().item()

    avg_loss = total_loss / len(loader)
    avg_acc = acc / len(loader.dataset)

    losses.append(avg_loss)
    accs.append(avg_acc)

    print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f} | Acc: {avg_acc:.4f}")

Training:   3%|▎         | 1/30 [00:02<01:14,  2.56s/it]

Epoch 1 | Loss: 0.8226 | Acc: 0.6060


Training:   7%|▋         | 2/30 [00:05<01:16,  2.72s/it]

Epoch 2 | Loss: 0.5840 | Acc: 0.7345


Training:  10%|█         | 3/30 [00:07<01:10,  2.62s/it]

Epoch 3 | Loss: 0.4984 | Acc: 0.7976


... (50 outputs left out)

Training:  97%|█████████▋| 29/30 [01:17<00:02,  2.81s/it]

Epoch 29 | Loss: 0.0776 | Acc: 0.9757


Training: 100%|██████████| 30/30 [01:19<00:00,  2.65s/it]

Epoch 30 | Loss: 0.0710 | Acc: 0.9777


In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(losses, label="Loss")
plt.plot(accs, label="Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Value")
plt.title("Training Loss & Accuracy")
plt.legend()
plt.show()

In [22]:
@torch.no_grad()
def inference(review, max_len):
    model.eval()

    # 1. preprocessamento
    review = process_text(review)
    tokens = tokenized(review)

    # 2. pad para o mesmo tamanho usado no treino
    if tokens.size(0) < max_len:
      pad_amount = max_len - tokens.size(0)
      tokens = torch.cat([
          tokens,
          torch.full((pad_amount,), words["<pad>"], dtype=torch.long, device=device)
      ])
    else:
        tokens = tokens[-max_len:]

    # 3. batch size = 1
    tokens = tokens.unsqueeze(0)

    # 4. forward
    logits = model(tokens)

    # 5. probabilidades e classe
    probs = torch.softmax(logits, dim=1)
    pred_class = torch.argmax(probs, dim=1).item()

    return {
        "logits": logits,
        "probs": probs,
        "pred": pred_class
    }

In [23]:
MAX_LENGTH = max([len(x) for x in reviews["Comment"]])
review = """Man, what a terrible movie.
The story makes no sense, the characters seem like robots,
and the editing is so messy it gives me a headache.
Seriously, I wish I had my time back."""
out = inference(review, MAX_LENGTH)
out["pred"]

0

In [24]:
review = """Wow, what a great movie!
The story is captivating from beginning to end,
the characters are super charismatic, and the soundtrack is amazing.
I left the theater with that big smile of someone who'd seen something truly special.
I highly recommend it!"""
out = inference(review, MAX_LENGTH)
out["pred"]

0